# TRANSFORMACIÓN DE DATOS

## IMPORTAR PAQUETES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

## IMPORTAR LOS DATOS

In [ ]:
cat = pd.read_pickle("../02_datos/03_Entrenamiento/cat_resultado_eda.pkl")
num = pd.read_pickle("../02_datos/03_Entrenamiento/num_resultado_eda.pkl")

## NUEVAS VARIABLES

En este caso ya nos vienen creadas:

* componentes de la fecha
* variables de calendario

Vamos a crear:

* las que identificamos en EDA
* lags
* ventanas móviles

Para esta fase necesitamos volver a unir en un solo dataframe.

In [ ]:
df = pd.concat([cat,num], axis = 1)
df

### Variable demanda intermitente

Esta variable va a identificar cuantos días seguidos han transcurrido con ventas cero.

La definiremos como que si los últimos n días han tenido cero ventas entonces hay rotura de stock.

Podemos crear varias cambiando el n.

Nos servirá para modelizar.

In [ ]:
def rotura_stock(ventas, n = 5):
    cero_ventas = pd.Series(np.where(ventas == 0,1,0))
    num_ceros = cero_ventas.rolling(n).sum()
    rotura_stock = np.where(num_ceros == n,1,0)
    return(rotura_stock)

In [ ]:
df = df.sort_values(by = ['store_id','item_id','date'])

In [ ]:
df['rotura_stock_3'] = df.groupby(['store_id','item_id']).ventas.transform(lambda x: rotura_stock(x, 3)).values

In [ ]:
df['rotura_stock_7'] = df.groupby(['store_id','item_id']).ventas.transform(lambda x: rotura_stock(x,7)).values

In [ ]:
df['rotura_stock_15'] = df.groupby(['store_id','item_id']).ventas.transform(lambda x: rotura_stock(x,15)).values

### Variables de lag

Vamos a crear lags sobre las siguientes variables:

* ventas: lags de 15 días
* sell_price: lags de 7 días
* rotura_stock: lag de un día

In [ ]:
def crear_lags(df, variable, num_lags = 7):
    
    #Crea el objeto dataframe
    lags = pd.DataFrame()
    
    #Crea todos los lags
    for cada in range(1,num_lags+1):
        lags[variable + '_lag_'+ str(cada)] = df[variable].shift(cada)
    
    #Devuelve el dataframe de lags
    return(lags)

In [ ]:
lags_sell_price_df = (df.groupby(['store_id', 'item_id'])
                        .apply(lambda x: crear_lags(df = x, variable = 'sell_price', num_lags= 7))
                        .reset_index()
                        .set_index('date'))


In [ ]:
lags_rotura_stock_3_df = (df.groupby(['store_id','item_id'])
                            .apply(lambda x: crear_lags(df = x, variable = 'rotura_stock_3', num_lags= 1))
                            .reset_index()
                            .set_index('date'))

In [ ]:
lags_rotura_stock_7_df = (df.groupby(['store_id','item_id'])
                            .apply(lambda x: crear_lags(df = x, variable = 'rotura_stock_7', num_lags= 1))
                            .reset_index()
                            .set_index('date'))

In [ ]:
lags_rotura_stock_15_df = (df.groupby(['store_id','item_id'])
                            .apply(lambda x: crear_lags(df = x, variable = 'rotura_stock_15', num_lags= 1))
                            .reset_index()
                            .set_index('date'))

In [ ]:
lags_ventas_df = (df.groupby(['store_id','item_id'])
                    .apply(lambda x: crear_lags(df = x, variable = 'ventas', num_lags= 15))
                    .reset_index()
                    .set_index('date'))

### Variables de ventanas móviles

Vamos a crear tres tipos de ventanas móviles sobre las ventas:

* mínimo móvil
* media móvil
* máximo móvil

Cada uno de ellos en el rango de 15 días.

In [ ]:
def min_movil(df, variable, num_periodos = 7):

    minm = pd.DataFrame()
    
    for cada in range(2,num_periodos+1):
        minm[variable + '_minm_' + str(cada)] = df[variable].shift(1).rolling(cada).min()
    
    #Devuelve el dataframe de lags
    return(minm)

In [ ]:
def media_movil(df, variable, num_periodos = 7):

    mm = pd.DataFrame()
    
    for cada in range(2,num_periodos+1):
        mm[variable + '_mm_' + str(cada)] = df[variable].shift(1).rolling(cada).mean()
    
    #Devuelve el dataframe de lags
    return(mm)

In [ ]:
def max_movil(df, variable, num_periodos = 7):

    maxm = pd.DataFrame()
    
    for cada in range(2,num_periodos+1):
        maxm[variable + '_maxm_' + str(cada)] = df[variable].shift(1).rolling(cada).max()
    
    #Devuelve el dataframe de lags
    return(maxm)

In [ ]:
min_movil_df = (df.groupby(['store_id','item_id'])
                  .apply(lambda x: min_movil(df = x, variable = 'ventas', num_periodos= 15))
                  .reset_index()
                  .set_index('date'))

In [ ]:
media_movil_df = (df.groupby(['store_id','item_id'])
                    .apply(lambda x: media_movil(df = x, variable = 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

In [ ]:
max_movil_df = (df.groupby(['store_id','item_id'])
                    .apply(lambda x: max_movil(df = x, variable = 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

## PREPARAR LOS DATASETS

### Unir todos los dataframes generados

In [ ]:
df_unido = pd.concat([df,
                      lags_sell_price_df,
                      lags_rotura_stock_3_df,
                      lags_rotura_stock_7_df,
                      lags_rotura_stock_15_df,
                      lags_ventas_df,
                      min_movil_df,
                      media_movil_df,
                      max_movil_df], axis = 1)

# Eliminar columnas duplicadas
df_unido = df_unido.loc[:,~df_unido.columns.duplicated()]
df_unido

### Eliminar los nulos que han generado las nuevas variables

In [ ]:
df_unido.dropna(inplace=True)

### Eliminar las variables que no vamos a necesitar para modelizar

In [ ]:
a_eliminar = ['d','wm_yr_wk','sell_price','rotura_stock_3','rotura_stock_7','rotura_stock_15']

In [ ]:
df_unido.drop(columns=a_eliminar, inplace=True)

### Identificar la target

In [ ]:
target = df_unido.ventas

### Separar num y cat

In [ ]:
cat = df_unido.select_dtypes(include='O')

In [ ]:
num = df_unido.select_dtypes(exclude='O')

## TRANSFORMACIÓN DE CATEGÓRICAS

### One Hot Encoding

#### Variables a aplicar OHE

In [ ]:
var_ohe = ['year',
          'month',
          'wday',
          'weekday',
          'event_name_1',
          'event_type_1'
        ]

#### Instanciar

In [ ]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output = False, handle_unknown='ignore')

#### Entrenar y aplicar

In [ ]:
cat_ohe = ohe.fit_transform(cat[var_ohe])

#### Guardar como dataframe

In [ ]:
cat_ohe = pd.DataFrame(cat_ohe, columns = ohe.get_feature_names_out(cat[var_ohe].columns))

### Target Encoding

#### Variables a aplicar TE

In [ ]:
var_te = ['year',
          'month',
          'wday',
          'weekday',
          'event_name_1',
          'event_type_1'
        ]

#### Instanciar

In [ ]:
# !conda install -c conda-forge category_encoders
# !pip install category_encoders

In [ ]:
from category_encoders import TargetEncoder
te = TargetEncoder(min_samples_leaf=100, return_df = False)

#### Entrenar y aplicar

In [ ]:
cat_te = te.fit_transform(cat[var_te], y = target)

#### Guardar como dataframe

In [ ]:
#Añadir sufijos a los nombres
nombres_te = [variable + '_te' for variable in var_te]

#Guardar como dataframe
cat_te = pd.DataFrame(cat_te, columns = nombres_te)

## UNIFICAR DATASETS TRANSFORMADOS

### Meter en una lista todos los dataframes generados

Rescatamos de df_unido las variables de segmentación.

In [ ]:
de_df_unido = df_unido[['store_id','item_id']].reset_index()

de_df_unido.head(2)

### Unir todos los dataframes

In [ ]:
dataframes = [de_df_unido, cat_ohe,cat_te,num.reset_index(drop=True)]

In [ ]:
df_tablon = pd.concat(dataframes, axis = 1)

df_tablon

## GUARDAR DATASET TRAS TRANSFORMACIÓN DE DATOS

En formato pickle para no perder las modificaciones de metadatos.

In [ ]:
#Guardar los archivos
df_tablon.to_pickle("../02_datos/03_Entrenamiento/df_tablon_transformado.pkl")